# 🤟 Task 4 — Sign Language Detection
**Dataset**: ASL Alphabet (Kaggle) — 29 classes (A-Z + space, delete, nothing)

**Model**: Custom CNN from scratch

**GUI**: PyQt5 — Image Upload + Real-time Webcam

**Time Restriction**: Model only operational from 6 PM to 10 PM

---
### Steps:
1. Mount Drive & Download Dataset
2. Preprocess & Load Data
3. Build Custom CNN
4. Train Model
5. Evaluate & Save
6. Launch PyQt5 GUI

In [ ]:
# ============================================================
# CELL 1 — Mount Drive & Download ASL Alphabet Dataset
# ============================================================
from google.colab import drive, files
import os

drive.mount('/content/drive')

# Install Kaggle
!pip install kaggle -q

# Upload kaggle.json if not already done
if not os.path.exists('/root/.kaggle/kaggle.json'):
    files.upload()
    os.makedirs('/root/.kaggle', exist_ok=True)
    !cp kaggle.json /root/.kaggle/
    !chmod 600 /root/.kaggle/kaggle.json

# Define path
asl_path = '/content/drive/MyDrive/Internship_Datasets/Task4_Sign_Language_Detection/ASL_Alphabet'
os.makedirs(asl_path, exist_ok=True)

print('Downloading ASL Alphabet dataset (~1 GB)...')
!kaggle datasets download -d grassknoted/asl-alphabet \
    -p "{asl_path}" --unzip

print('Dataset downloaded!')
print(f'Path: {asl_path}')

In [ ]:
# ============================================================
# CELL 2 — Install Libraries & Imports
# ============================================================
!pip install tensorflow opencv-python-headless pyqt5 pillow matplotlib scikit-learn -q

import os
import numpy as np
import matplotlib.pyplot as plt
import cv2
from datetime import datetime
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix

import tensorflow as tf
from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.layers import (
    Conv2D, MaxPooling2D, BatchNormalization,
    Dropout, Dense, Flatten, GlobalAveragePooling2D
)
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from tensorflow.keras.preprocessing.image import ImageDataGenerator

print(f'TensorFlow version: {tf.__version__}')
print(f'GPU available: {len(tf.config.list_physical_devices("GPU")) > 0}')

In [ ]:
# ============================================================
# CELL 3 — Verify Dataset Structure
# ============================================================
import os

asl_path = '/content/drive/MyDrive/Internship_Datasets/Task4_Sign_Language_Detection/ASL_Alphabet'

# Find the asl_alphabet_train folder
train_dir = None
for root, dirs, files_list in os.walk(asl_path):
    if 'asl_alphabet_train' in dirs:
        train_dir = os.path.join(root, 'asl_alphabet_train', 'asl_alphabet_train')
        break
    for d in dirs:
        sub = os.path.join(root, d)
        sub_items = os.listdir(sub)
        if len(sub_items) >= 25:
            train_dir = sub
            break

if train_dir is None:
    # Try direct path
    train_dir = os.path.join(asl_path, 'asl_alphabet_train', 'asl_alphabet_train')

print(f'Train directory: {train_dir}')
classes = sorted(os.listdir(train_dir))
print(f'Number of classes: {len(classes)}')
print(f'Classes: {classes}')

# Count images per class
total = 0
for cls in classes[:5]:
    count = len(os.listdir(os.path.join(train_dir, cls)))
    print(f'  {cls}: {count} images')
    total += count
print(f'  ... (showing first 5 classes)')

In [ ]:
# ============================================================
# CELL 4 — Load & Preprocess Dataset
# ============================================================
IMG_SIZE = 64       # Resize to 64x64
BATCH_SIZE = 64
MAX_PER_CLASS = 1000  # Use 1000 images per class to save time (87k total otherwise)

print(f'Loading images (max {MAX_PER_CLASS} per class)...')

images = []
labels = []

classes = sorted(os.listdir(train_dir))
print(f'Classes ({len(classes)}): {classes}')

for cls in classes:
    cls_path = os.path.join(train_dir, cls)
    img_files = os.listdir(cls_path)[:MAX_PER_CLASS]
    loaded = 0
    for img_file in img_files:
        img_path = os.path.join(cls_path, img_file)
        img = cv2.imread(img_path)
        if img is not None:
            img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            images.append(img)
            labels.append(cls)
            loaded += 1
    print(f'  Loaded {loaded} images for class: {cls}')

images = np.array(images, dtype='float32') / 255.0
print(f'\nTotal images loaded: {len(images)}')
print(f'Image shape: {images[0].shape}')

# Encode labels
le = LabelEncoder()
labels_encoded = le.fit_transform(labels)
labels_cat = to_categorical(labels_encoded, num_classes=len(classes))
print(f'Label shape: {labels_cat.shape}')

In [ ]:
# ============================================================
# CELL 5 — Visualize Sample Images
# ============================================================
fig, axes = plt.subplots(4, 8, figsize=(16, 8))
fig.suptitle('ASL Alphabet — Sample Images', fontsize=14)

for i, cls in enumerate(classes[:29]):
    ax = axes[i // 8][i % 8]
    idx = labels.index(cls)
    ax.imshow(images[idx])
    ax.set_title(cls, fontsize=8)
    ax.axis('off')

# Hide empty subplots
for j in range(len(classes), 32):
    axes[j // 8][j % 8].axis('off')

plt.tight_layout()
plt.savefig('/content/drive/MyDrive/Internship_Datasets/Task4_Sign_Language_Detection/sample_images.png')
plt.show()
print('Sample images saved!')

In [ ]:
# ============================================================
# CELL 6 — Train/Val/Test Split
# ============================================================
X_train, X_temp, y_train, y_temp = train_test_split(
    images, labels_cat, test_size=0.2, random_state=42, stratify=labels_encoded
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=42
)

print(f'Train set:      {X_train.shape[0]} images')
print(f'Validation set: {X_val.shape[0]} images')
print(f'Test set:       {X_test.shape[0]} images')

# Data Augmentation
datagen = ImageDataGenerator(
    rotation_range=15,
    width_shift_range=0.1,
    height_shift_range=0.1,
    horizontal_flip=False,  # ASL is not symmetric
    zoom_range=0.1,
    shear_range=0.1
)
datagen.fit(X_train)
print('Data augmentation configured!')

In [ ]:
# ============================================================
# CELL 7 — Build Custom CNN from Scratch
# ============================================================
NUM_CLASSES = len(classes)

def build_asl_cnn(input_shape=(64, 64, 3), num_classes=29):
    model = Sequential([
        # --- Conv Block 1 ---
        Conv2D(32, (3,3), activation='relu', padding='same', input_shape=input_shape),
        BatchNormalization(),
        Conv2D(32, (3,3), activation='relu', padding='same'),
        BatchNormalization(),
        MaxPooling2D(2,2),
        Dropout(0.25),

        # --- Conv Block 2 ---
        Conv2D(64, (3,3), activation='relu', padding='same'),
        BatchNormalization(),
        Conv2D(64, (3,3), activation='relu', padding='same'),
        BatchNormalization(),
        MaxPooling2D(2,2),
        Dropout(0.25),

        # --- Conv Block 3 ---
        Conv2D(128, (3,3), activation='relu', padding='same'),
        BatchNormalization(),
        Conv2D(128, (3,3), activation='relu', padding='same'),
        BatchNormalization(),
        MaxPooling2D(2,2),
        Dropout(0.25),

        # --- Conv Block 4 ---
        Conv2D(256, (3,3), activation='relu', padding='same'),
        BatchNormalization(),
        Conv2D(256, (3,3), activation='relu', padding='same'),
        BatchNormalization(),
        GlobalAveragePooling2D(),
        Dropout(0.4),

        # --- Classifier Head ---
        Dense(512, activation='relu'),
        BatchNormalization(),
        Dropout(0.5),
        Dense(256, activation='relu'),
        Dropout(0.3),
        Dense(num_classes, activation='softmax')
    ])
    return model

model = build_asl_cnn(input_shape=(IMG_SIZE, IMG_SIZE, 3), num_classes=NUM_CLASSES)
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)
model.summary()

In [ ]:
# ============================================================
# CELL 8 — Train the Model
# ============================================================
model_save_path = '/content/drive/MyDrive/Internship_Datasets/Task4_Sign_Language_Detection/asl_cnn_model.h5'

callbacks = [
    EarlyStopping(monitor='val_accuracy', patience=8, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=4, min_lr=1e-6, verbose=1),
    ModelCheckpoint(model_save_path, monitor='val_accuracy', save_best_only=True, verbose=1)
]

EPOCHS = 30

print(f'Training for up to {EPOCHS} epochs...')
history = model.fit(
    datagen.flow(X_train, y_train, batch_size=BATCH_SIZE),
    epochs=EPOCHS,
    validation_data=(X_val, y_val),
    callbacks=callbacks,
    verbose=1
)

print(f'\nModel saved to: {model_save_path}')

In [ ]:
# ============================================================
# CELL 9 — Plot Training History
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Accuracy
axes[0].plot(history.history['accuracy'], label='Train Accuracy', color='blue')
axes[0].plot(history.history['val_accuracy'], label='Val Accuracy', color='orange')
axes[0].set_title('Model Accuracy')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Loss
axes[1].plot(history.history['loss'], label='Train Loss', color='blue')
axes[1].plot(history.history['val_loss'], label='Val Loss', color='orange')
axes[1].set_title('Model Loss')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('/content/drive/MyDrive/Internship_Datasets/Task4_Sign_Language_Detection/training_history.png')
plt.show()
print('Training history saved!')

In [ ]:
# ============================================================
# CELL 10 — Evaluate on Test Set
# ============================================================
test_loss, test_acc = model.evaluate(X_test, y_test, verbose=0)
print(f'Test Accuracy: {test_acc * 100:.2f}%')
print(f'Test Loss:     {test_loss:.4f}')

# Predictions
y_pred = model.predict(X_test)
y_pred_classes = np.argmax(y_pred, axis=1)
y_true_classes = np.argmax(y_test, axis=1)

# Classification report
print('\nClassification Report:')
print(classification_report(
    y_true_classes, y_pred_classes,
    target_names=le.classes_
))

# Save label encoder classes
import json
classes_path = '/content/drive/MyDrive/Internship_Datasets/Task4_Sign_Language_Detection/classes.json'
with open(classes_path, 'w') as f:
    json.dump(list(le.classes_), f)
print(f'Classes saved to: {classes_path}')

In [ ]:
# ============================================================
# CELL 11 — Confusion Matrix
# ============================================================
import seaborn as sns

cm = confusion_matrix(y_true_classes, y_pred_classes)
plt.figure(figsize=(20, 16))
sns.heatmap(
    cm, annot=True, fmt='d', cmap='Blues',
    xticklabels=le.classes_,
    yticklabels=le.classes_
)
plt.title('Confusion Matrix — ASL Sign Language CNN', fontsize=14)
plt.xlabel('Predicted')
plt.ylabel('True')
plt.tight_layout()
plt.savefig('/content/drive/MyDrive/Internship_Datasets/Task4_Sign_Language_Detection/confusion_matrix.png')
plt.show()
print('Confusion matrix saved!')

In [ ]:
# ============================================================
# CELL 12 — PyQt5 GUI (Run Locally, NOT in Colab)
# ============================================================
# NOTE: Save this cell as a separate .py file and run on your
# local machine after downloading the model (.h5) and classes.json
# ============================================================

gui_code = '''
import sys
import cv2
import json
import numpy as np
from datetime import datetime
from tensorflow.keras.models import load_model
from PyQt5.QtWidgets import (
    QApplication, QMainWindow, QWidget, QVBoxLayout, QHBoxLayout,
    QPushButton, QLabel, QFileDialog, QFrame, QMessageBox, QProgressBar
)
from PyQt5.QtCore import Qt, QTimer, QThread, pyqtSignal
from PyQt5.QtGui import QImage, QPixmap, QFont, QColor, QPalette
from PIL import Image

# ── Config ──────────────────────────────────────────────────
MODEL_PATH  = "asl_cnn_model.h5"
CLASSES_PATH = "classes.json"
IMG_SIZE    = 64
ACTIVE_START = 18   # 6 PM
ACTIVE_END   = 22   # 10 PM

# ── Time Check ──────────────────────────────────────────────
def is_operational():
    hour = datetime.now().hour
    return ACTIVE_START <= hour < ACTIVE_END

# ── Preprocess ──────────────────────────────────────────────
def preprocess(img_bgr):
    img = cv2.resize(img_bgr, (IMG_SIZE, IMG_SIZE))
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img = img.astype("float32") / 255.0
    return np.expand_dims(img, axis=0)

# ── Webcam Thread ────────────────────────────────────────────
class WebcamThread(QThread):
    frame_signal = pyqtSignal(np.ndarray)

    def __init__(self):
        super().__init__()
        self.running = False

    def run(self):
        self.running = True
        cap = cv2.VideoCapture(0)
        while self.running:
            ret, frame = cap.read()
            if ret:
                self.frame_signal.emit(frame)
        cap.release()

    def stop(self):
        self.running = False
        self.wait()

# ── Main Window ──────────────────────────────────────────────
class SignLanguageApp(QMainWindow):
    def __init__(self):
        super().__init__()
        self.setWindowTitle("ASL Sign Language Detection")
        self.setMinimumSize(900, 650)
        self.model = load_model(MODEL_PATH)
        with open(CLASSES_PATH) as f:
            self.classes = json.load(f)
        self.webcam_thread = None
        self.init_ui()
        self.time_timer = QTimer()
        self.time_timer.timeout.connect(self.update_time_status)
        self.time_timer.start(5000)
        self.update_time_status()

    def init_ui(self):
        central = QWidget()
        self.setCentralWidget(central)
        main_layout = QVBoxLayout(central)
        main_layout.setSpacing(12)
        main_layout.setContentsMargins(16, 16, 16, 16)

        # Title
        title = QLabel("ASL Sign Language Detection")
        title.setFont(QFont("Arial", 20, QFont.Bold))
        title.setAlignment(Qt.AlignCenter)
        title.setStyleSheet("color: #2c3e50; padding: 8px;")
        main_layout.addWidget(title)

        # Time status bar
        self.time_label = QLabel()
        self.time_label.setAlignment(Qt.AlignCenter)
        self.time_label.setFont(QFont("Arial", 11))
        self.time_label.setFixedHeight(36)
        self.time_label.setStyleSheet("border-radius: 6px; padding: 4px 12px;")
        main_layout.addWidget(self.time_label)

        # Content row
        content = QHBoxLayout()
        content.setSpacing(12)

        # Left — Image/Video feed
        left_panel = QVBoxLayout()
        self.video_label = QLabel("No image loaded")
        self.video_label.setFixedSize(480, 380)
        self.video_label.setAlignment(Qt.AlignCenter)
        self.video_label.setStyleSheet("""
            border: 2px dashed #bdc3c7;
            border-radius: 8px;
            background: #f8f9fa;
            color: #7f8c8d;
            font-size: 14px;
        """)
        left_panel.addWidget(self.video_label)

        # Buttons row
        btn_row = QHBoxLayout()
        self.upload_btn = QPushButton(" Upload Image")
        self.upload_btn.setFixedHeight(42)
        self.upload_btn.setFont(QFont("Arial", 11))
        self.upload_btn.setStyleSheet("""
            QPushButton {
                background: #3498db; color: white;
                border-radius: 6px; font-weight: bold;
            }
            QPushButton:hover { background: #2980b9; }
            QPushButton:disabled { background: #bdc3c7; }
        """)
        self.upload_btn.clicked.connect(self.upload_image)

        self.webcam_btn = QPushButton(" Start Webcam")
        self.webcam_btn.setFixedHeight(42)
        self.webcam_btn.setFont(QFont("Arial", 11))
        self.webcam_btn.setStyleSheet("""
            QPushButton {
                background: #27ae60; color: white;
                border-radius: 6px; font-weight: bold;
            }
            QPushButton:hover { background: #229954; }
            QPushButton:disabled { background: #bdc3c7; }
        """)
        self.webcam_btn.clicked.connect(self.toggle_webcam)

        btn_row.addWidget(self.upload_btn)
        btn_row.addWidget(self.webcam_btn)
        left_panel.addLayout(btn_row)
        content.addLayout(left_panel)

        # Right — Results panel
        right_panel = QVBoxLayout()
        right_panel.setSpacing(10)

        result_title = QLabel("Prediction")
        result_title.setFont(QFont("Arial", 14, QFont.Bold))
        result_title.setStyleSheet("color: #2c3e50;")
        right_panel.addWidget(result_title)

        # Predicted letter box
        self.pred_letter = QLabel("-")
        self.pred_letter.setFont(QFont("Arial", 72, QFont.Bold))
        self.pred_letter.setAlignment(Qt.AlignCenter)
        self.pred_letter.setFixedSize(180, 160)
        self.pred_letter.setStyleSheet("""
            background: #ecf0f1;
            border-radius: 12px;
            color: #2c3e50;
            border: 2px solid #bdc3c7;
        """)
        right_panel.addWidget(self.pred_letter, alignment=Qt.AlignCenter)

        # Confidence
        conf_label = QLabel("Confidence")
        conf_label.setFont(QFont("Arial", 11))
        conf_label.setStyleSheet("color: #7f8c8d;")
        right_panel.addWidget(conf_label)

        self.conf_bar = QProgressBar()
        self.conf_bar.setRange(0, 100)
        self.conf_bar.setValue(0)
        self.conf_bar.setFixedHeight(22)
        self.conf_bar.setStyleSheet("""
            QProgressBar { border-radius: 6px; background: #ecf0f1; }
            QProgressBar::chunk { background: #27ae60; border-radius: 6px; }
        """)
        right_panel.addWidget(self.conf_bar)

        self.conf_text = QLabel("0%")
        self.conf_text.setFont(QFont("Arial", 13, QFont.Bold))
        self.conf_text.setAlignment(Qt.AlignCenter)
        self.conf_text.setStyleSheet("color: #27ae60;")
        right_panel.addWidget(self.conf_text)

        # Top 3 predictions
        top3_label = QLabel("Top 3 Predictions")
        top3_label.setFont(QFont("Arial", 11, QFont.Bold))
        top3_label.setStyleSheet("color: #2c3e50; margin-top: 8px;")
        right_panel.addWidget(top3_label)

        self.top3_labels = []
        for i in range(3):
            lbl = QLabel(f"{i+1}. -")
            lbl.setFont(QFont("Arial", 11))
            lbl.setStyleSheet("color: #555; padding: 4px 8px; background: #f0f0f0; border-radius: 4px;")
            right_panel.addWidget(lbl)
            self.top3_labels.append(lbl)

        right_panel.addStretch()

        # Current time
        self.clock_label = QLabel()
        self.clock_label.setFont(QFont("Arial", 10))
        self.clock_label.setStyleSheet("color: #95a5a6;")
        self.clock_label.setAlignment(Qt.AlignCenter)
        right_panel.addWidget(self.clock_label)

        content.addLayout(right_panel)
        main_layout.addLayout(content)

        # Clock update
        self.clock_timer = QTimer()
        self.clock_timer.timeout.connect(self.update_clock)
        self.clock_timer.start(1000)
        self.update_clock()

    def update_clock(self):
        now = datetime.now().strftime("%I:%M:%S %p — %A, %d %b %Y")
        self.clock_label.setText(now)

    def update_time_status(self):
        if is_operational():
            self.time_label.setText(" Model is ACTIVE (6 PM – 10 PM)")
            self.time_label.setStyleSheet("""
                background: #d5f5e3; color: #1e8449;
                border-radius: 6px; padding: 4px 12px; font-weight: bold;
            """)
            self.upload_btn.setEnabled(True)
            self.webcam_btn.setEnabled(True)
        else:
            self.time_label.setText(" Model is INACTIVE — Only available from 6 PM to 10 PM")
            self.time_label.setStyleSheet("""
                background: #fadbd8; color: #922b21;
                border-radius: 6px; padding: 4px 12px; font-weight: bold;
            """)
            self.upload_btn.setEnabled(False)
            self.webcam_btn.setEnabled(False)
            if self.webcam_thread and self.webcam_thread.running:
                self.webcam_thread.stop()

    def predict_frame(self, frame):
        processed = preprocess(frame)
        preds = self.model.predict(processed, verbose=0)[0]
        top3_idx = np.argsort(preds)[::-1][:3]
        top_class = self.classes[top3_idx[0]]
        top_conf  = preds[top3_idx[0]] * 100

        self.pred_letter.setText(top_class)
        self.conf_bar.setValue(int(top_conf))
        self.conf_text.setText(f"{top_conf:.1f}%")

        for i, idx in enumerate(top3_idx):
            self.top3_labels[i].setText(f"{i+1}. {self.classes[idx]}  ({preds[idx]*100:.1f}%)")

        # Color code confidence
        if top_conf >= 80:
            color = "#27ae60"
        elif top_conf >= 50:
            color = "#f39c12"
        else:
            color = "#e74c3c"
        self.conf_bar.setStyleSheet(f"""
            QProgressBar {{ border-radius: 6px; background: #ecf0f1; }}
            QProgressBar::chunk {{ background: {color}; border-radius: 6px; }}
        """)
        self.conf_text.setStyleSheet(f"color: {color}; font-weight: bold; font-size: 13px;")

    def show_frame(self, frame):
        if not is_operational():
            return
        self.predict_frame(frame)
        rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        h, w, ch = rgb.shape
        qt_img = QImage(rgb.data, w, h, ch * w, QImage.Format_RGB888)
        pixmap = QPixmap.fromImage(qt_img).scaled(
            480, 380, Qt.KeepAspectRatio, Qt.SmoothTransformation
        )
        self.video_label.setPixmap(pixmap)

    def upload_image(self):
        if not is_operational():
            QMessageBox.warning(self, "Inactive", "Model is only active from 6 PM to 10 PM!")
            return
        path, _ = QFileDialog.getOpenFileName(
            self, "Select Image", "",
            "Images (*.png *.jpg *.jpeg *.bmp)"
        )
        if path:
            frame = cv2.imread(path)
            if frame is not None:
                self.predict_frame(frame)
                rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                h, w, ch = rgb.shape
                qt_img = QImage(rgb.data, w, h, ch * w, QImage.Format_RGB888)
                pixmap = QPixmap.fromImage(qt_img).scaled(
                    480, 380, Qt.KeepAspectRatio, Qt.SmoothTransformation
                )
                self.video_label.setPixmap(pixmap)

    def toggle_webcam(self):
        if not is_operational():
            QMessageBox.warning(self, "Inactive", "Model is only active from 6 PM to 10 PM!")
            return
        if self.webcam_thread and self.webcam_thread.running:
            self.webcam_thread.stop()
            self.webcam_thread = None
            self.webcam_btn.setText(" Start Webcam")
            self.webcam_btn.setStyleSheet("""
                QPushButton { background: #27ae60; color: white; border-radius: 6px; font-weight: bold; }
                QPushButton:hover { background: #229954; }
            """)
            self.video_label.setText("Webcam stopped")
        else:
            self.webcam_thread = WebcamThread()
            self.webcam_thread.frame_signal.connect(self.show_frame)
            self.webcam_thread.start()
            self.webcam_btn.setText(" Stop Webcam")
            self.webcam_btn.setStyleSheet("""
                QPushButton { background: #e74c3c; color: white; border-radius: 6px; font-weight: bold; }
                QPushButton:hover { background: #c0392b; }
            """)

    def closeEvent(self, event):
        if self.webcam_thread:
            self.webcam_thread.stop()
        event.accept()

if __name__ == "__main__":
    app = QApplication(sys.argv)
    app.setStyle("Fusion")
    window = SignLanguageApp()
    window.show()
    sys.exit(app.exec_())
'''

# Save GUI script to Drive
gui_path = '/content/drive/MyDrive/Internship_Datasets/Task4_Sign_Language_Detection/asl_gui.py'
with open(gui_path, 'w') as f:
    f.write(gui_code)

print('GUI script saved to Drive!')
print(f'Path: {gui_path}')
print()
print('To run the GUI locally:')
print('  1. Download asl_cnn_model.h5, classes.json and asl_gui.py from Drive')
print('  2. Put all 3 files in the same folder')
print('  3. Run: pip install tensorflow pyqt5 opencv-python pillow')
print('  4. Run: python asl_gui.py')
print('  NOTE: GUI only works between 6 PM and 10 PM!')

In [ ]:
# ============================================================
# CELL 13 — Test Model: Upload Your Own Image
# ============================================================
from google.colab import files as colab_files
from tensorflow.keras.models import load_model
import numpy as np
import cv2
import json
import matplotlib.pyplot as plt

# --- Load model & classes ---
model_path   = "/content/drive/MyDrive/Internship_Datasets/Task4_Sign_Language_Detection/asl_cnn_model.h5"
classes_path = "/content/drive/MyDrive/Internship_Datasets/Task4_Sign_Language_Detection/classes.json"

model = load_model(model_path)
with open(classes_path, "r") as f:
    raw = json.load(f)

# Handle both list and dict formats of classes.json
if isinstance(raw, list):
    classes = {str(i): label for i, label in enumerate(raw)}
else:
    classes = raw

print("Model and classes loaded!")
print("Total classes: " + str(len(classes)))
print("Classes: " + str(list(classes.values())))
print()

# --- Upload your image ---
print("Please upload your hand sign image...")
uploaded = colab_files.upload()

for filename, data in uploaded.items():
    with open(filename, "wb") as f:
        f.write(data)

    img_bgr = cv2.imread(filename)
    if img_bgr is None:
        print("Could not read: " + filename)
        continue

    # Preprocess
    img_resized = cv2.resize(img_bgr, (64, 64))
    img_rgb     = cv2.cvtColor(img_resized, cv2.COLOR_BGR2RGB)
    img_norm    = img_rgb.astype("float32") / 255.0
    img_input   = np.expand_dims(img_norm, axis=0)

    # Predict
    preds     = model.predict(img_input, verbose=0)[0]
    top3_idx  = np.argsort(preds)[::-1][:3]
    top_label = classes[str(top3_idx[0])]
    top_conf  = float(preds[top3_idx[0]]) * 100

    # --- Show image + prediction ---
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    fig.suptitle("ASL Sign Prediction - " + filename, fontsize=14, fontweight="bold")

    # Left: uploaded image
    display_img = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    axes[0].imshow(display_img)
    color = "green" if top_conf >= 80 else ("orange" if top_conf >= 50 else "red")
    axes[0].set_title("Predicted: " + top_label + "  (" + str(round(top_conf, 1)) + "%)",
                      fontsize=13, fontweight="bold", color=color)
    axes[0].axis("off")

    # Right: top-3 bar chart
    top3_labels = [classes[str(i)] for i in top3_idx]
    top3_confs  = [float(preds[i]) * 100 for i in top3_idx]
    bar_colors  = ["#2ecc71", "#f39c12", "#e74c3c"]
    axes[1].barh(top3_labels[::-1], top3_confs[::-1], color=bar_colors[::-1])
    axes[1].set_xlim(0, 100)
    axes[1].set_xlabel("Confidence (%)", fontsize=11)
    axes[1].set_title("Top 3 Predictions", fontsize=13, fontweight="bold")
    for i, (lbl, conf) in enumerate(zip(top3_labels[::-1], top3_confs[::-1])):
        axes[1].text(conf + 1, i, str(round(conf, 1)) + "%", va="center", fontsize=11, fontweight="bold")

    plt.tight_layout()
    plt.show()

    print("--------------------------------------------------")
    print("Result for: " + filename)
    print("   Predicted Sign : " + top_label)
    print("   Confidence     : " + str(round(top_conf, 2)) + "%")
    print("   Top 3 Predictions:")
    for rank, idx in enumerate(top3_idx, 1):
        label = classes[str(idx)]
        conf  = round(float(preds[idx]) * 100, 2)
        print("      " + str(rank) + ". " + label.ljust(10) + " - " + str(conf) + "%")
    print("--------------------------------------------------")
